# Minilink showcase

[![Open In Colab](https://colab.research.google.com/assets/colab-badge.svg)](https://colab.research.google.com/github/alx87grd/minilink/blob/main/examples/notebooks/showcase/minilink.ipynb)

Minilink is a Python-native block-diagram framework for modeling, simulating,
optimizing, and visualizing dynamical systems. The promise: **the code reads
like the textbook math**, and **everything is a `System`** — plants,
controllers, sources, and full diagrams alike.

This notebook is a short product tour, from a single plant to hybrid MPC:

1. [Quick start](#1.-Quick-start) — plant, open loop, closed loop
2. [Everything is a System](#2.-Everything-is-a-System) — ports and trajectories
3. [Write your own model](#3.-Write-your-own-model) — textbook `f`
4. [Parameters](#4.-Parameters) — tune via `.params`
5. [Compose diagrams](#5.-Compose-diagrams) — `+` and `.autowire`
6. [Plant and control catalogs](#6.-Plant-and-control-catalogs) — CartPole and LQR
7. [Animation and custom graphics](#7.-Animation-and-custom-graphics) — renderers + primitives
8. [Optimization](#8.-Optimization) — `MathematicalProgram`
9. [Compiled execution and JAX](#9.-Compiled-execution-and-JAX) — compile + autodiff
10. [Trajectory optimization](#10.-Trajectory-optimization) — swing-up
11. [Hybrid and MPC](#11.-Hybrid-and-MPC) — `mpc @ plant`

**See also**

- [`showcase/jax.ipynb`](jax.ipynb) — why stateless `f` + autodiff + compile
- [`intro/`](../intro/) — per-module API intros (`00_core` … `10_graphical`)
- [`applications/`](../applications/) — full MPC circuit and car TrajOpt labs

**Run it locally** with the conda env `minilink` (see the repo README Install),
or open on Colab — the setup cell detects Colab and clones the repo automatically.


In [ ]:
# Local conda: minilink already installed. Colab: clone + path + meshcat.
import importlib.util
import os
import sys

if "google.colab" in sys.modules:
    get_ipython().run_line_magic("matplotlib", "inline")
    get_ipython().system("git clone https://github.com/alx87grd/minilink")
    sys.path.insert(0, "/content/minilink")
    get_ipython().system("pip install -q meshcat")

_NOTEBOOK_SMOKE = os.environ.get("MINILINK_NOTEBOOK_SMOKE") == "1"
_OPTIMIZER_METHOD = (
    "scipy_slsqp"
    if _NOTEBOOK_SMOKE or importlib.util.find_spec("cyipopt") is None
    else "ipopt"
)


## 1. Quick start

Start with a catalog plant and its defaults. Simulate and visualize the free
response, wire an open-loop source, then close the loop with a controller.


### A plant by itself

Every minilink model is a **`System`**: named input/output **ports** and a state
vector `x`. The contract is one **dynamics** equation plus one map **per output
port**:

    dx_dt = f(x, u, t)          # how the state evolves
    y     = h(x, u, t)          # one equation per output port

Catalog plants ship with sensible defaults — construct, set `x0` if you want,
and display the object (its port diagram shows the boundary signals).


In [ ]:
import numpy as np

from minilink.dynamics.catalog.pendulum.pendulum import Pendulum

sys = Pendulum()
sys


Visualize **f**: `plot_phase_plane()` draws the vector field of the dynamics —
how the state would move at each point.


In [ ]:
sys.plot_phase_plane()


`compute_trajectory()` integrates **f** and caches the result on the system
(for plotting and animation next).


In [ ]:
sys.x0[0] = 2.0  # initial angle [rad]
sys.compute_trajectory(tf=10.0)


`plot_trajectory()` stacks labeled, unit-aware time signals.


In [ ]:
sys.plot_trajectory()


`plot_phase_plane()` again — now with the simulated path overlaid on the
vector field.


In [ ]:
sys.plot_phase_plane()


`animate()` replays the cached motion. **plotly** is a good default in
notebooks (fast HTML animation).


In [ ]:
sys.animate(renderer="plotly")


### Open loop: `source >> plant`

`>>` chains a step source into the plant input. The diagram is itself a
`System`. Uncomment the lines below to integrate and plot this open loop.


In [ ]:
from minilink.blocks.sources import Step

step = Step(final_value=np.array([2.0]), step_time=5.0)
diagram = step >> sys
diagram

# diagram.compute_trajectory()
# diagram.plot_trajectory()


### Closed loop: `controller @ plant`

`@` builds the standard feedback loop. Pass gains to the controller
constructor (`Kp`, `Kd`); the reference port `r` stays at its nominal zero here.


In [ ]:
from minilink.control.impedance import ImpedanceController

ctl = ImpedanceController(Kp=100.0, Kd=10.0)
diagram2 = step >> ctl @ sys
diagram2


In [ ]:
diagram2.compute_trajectory(tf=10.0)


In [ ]:
diagram2.plot_trajectory()


In [ ]:
diagram2.animate(renderer="plotly")


## 2. Everything is a System

A `System` owns its equations (`f`, `h`), named **ports**, and initial condition.
Trajectories live in the returned `Trajectory` — not hidden inside the object.


In [ ]:
print("Plant:   ", sys.name)
print("states:  ", sys.n, sys.state.labels, sys.state.units)
print("inputs:  ", sys.m, list(sys.inputs))
print("outputs: ", list(sys.outputs))
print("Diagram: ", diagram2.n, "states")


## 3. Write your own model

Subclass `DynamicSystem` and write $\dot x = f(x,u,t)$ like the textbook —
coefficients as plain locals in `f`. `output_dim=2` creates the standard `y`
port so `>>` can wire the diagram.


In [ ]:
from minilink.core.system import DynamicSystem


class MassSpringDamper(DynamicSystem):
    # m x'' + c x' + k x = u

    def __init__(self):
        super().__init__(n=2, input_dim=1, output_dim=2)

    def f(self, x, u, t=0, params=None):
        # m \ddot{p} + c \dot{p} + k p = f
        m = 1.0  # mass [kg]
        k = 4.0  # stiffness [N/m]
        c = 0.3  # damping [N.s/m]
        p = x[0]  # position [m]
        v = x[1]  # velocity [m/s]
        f = u[0]  # applied force [N]
        a = (f - c * v - k * p) / m  # acceleration [m/s^2]
        return np.array([v, a])


msd = MassSpringDamper()
msd.x0[0] = 1.0
msd_chain = Step(final_value=np.array([10.0]), step_time=2.0) >> msd


In [ ]:
msd_chain.compute_trajectory(tf=20.0, n_steps=1001)


In [ ]:
msd_chain.plot_trajectory()


## 4. Parameters

Same ODE as §3, but `m`, `k`, `c` live in `self.params` and are read from
`f`'s `params` argument. Contract: `params is None` means “use
`self.params`”; any other dict overrides. Catalog blocks use the same
`.params[...]` pattern (see §6). For derivatives with respect to parameters,
see [`showcase/jax.ipynb`](jax.ipynb).


In [ ]:
class MassSpringDamperParams(DynamicSystem):
    # m \ddot{p} + c \dot{p} + k p = f  (same ODE as §3)

    def __init__(self):
        super().__init__(n=2, input_dim=1, output_dim=2)
        self.params = {"m": 1.0, "k": 4.0, "c": 0.3}

    def f(self, x, u, t=0, params=None):
        if params is None:
            params = self.params
        m = params["m"]  # mass [kg]
        k = params["k"]  # stiffness [N/m]
        c = params["c"]  # damping [N.s/m]
        p = x[0]  # position [m]
        v = x[1]  # velocity [m/s]
        f = u[0]  # applied force [N]
        a = (f - c * v - k * p) / m  # acceleration [m/s^2]
        return np.array([v, a])


msd_p = MassSpringDamperParams()
msd_p.params["c"] = 1.8  # retune damping without rewriting f
msd_p.x0[0] = 1.0

msd_p_chain = Step(final_value=np.array([10.0]), step_time=2.0) >> msd_p


In [ ]:
msd_p_chain.compute_trajectory(tf=20.0, n_steps=1001)


In [ ]:
msd_p_chain.plot_trajectory()


## 5. Compose diagrams

| Shortcut | Meaning |
| --- | --- |
| `a + b + c` | add subsystems without wiring |
| `source >> plant` | chain output to input (§1) |
| `controller @ plant` | simple feedback diagram (§1) |
| `.autowire(strict=True)` | connect matching named ports |

When port names are unambiguous, autowire builds the loop in one line:


In [ ]:
auto = Step() + ImpedanceController() + Pendulum()
auto.autowire(strict=True)
auto


## 6. Plant and control catalogs

`minilink.dynamics.catalog` ships ready-to-use plants; `minilink.control` ships
controllers and design factories. Below: forced cart-pole response, then LQR
stabilization.


In [ ]:
from minilink.dynamics.catalog.pendulum.cartpole import CartPole

cartpole = CartPole()
# cartpole.compute_forced(lambda t: np.array([3.0 * np.sin(2.0 * t)]))


In [ ]:
# cartpole.plot_trajectory()


### Control catalog

Controllers are `System` blocks with standard ports (`r`, `y`, `u`).
`lqr_at_operating_point` linearizes at an equilibrium and returns a
`StateFeedbackController` ready for `@`.


In [ ]:
from minilink.control.lqr import lqr_at_operating_point

plant = CartPole()
x_bar = np.array([0.0, np.pi, 0.0, 0.0])
Q = np.diag([1.0, 1.0, 1.0, 1.0])
R = np.array([[1.0]])

lqr_ctl = lqr_at_operating_point(plant, x_bar, Q, R)
lqr_loop = lqr_ctl @ plant

plant.x0 = np.array([-3.0, np.pi - 0.3, 0.0, 0.0])


In [ ]:
lqr_loop.compute_trajectory(tf=8.0)


In [ ]:
lqr_loop.plot_trajectory()


In [ ]:
lqr_loop.animate()


## 7. Animation and custom graphics

`animate()` draws named **frames** of geometric **primitives** (boxes, rods,
springs, …). The same call works with several renderers — pick one that fits
the environment:

| Renderer | Typical use |
| --- | --- |
| `plotly` | interactive HTML in notebooks |
| `meshcat` | 3D in the browser |
| `matplotlib` | inline / Agg / saved figures |
| `pygame` | native window (local scripts) |

Catalog plants already ship a skin. For a custom plant you only need a few
hooks: static geometry (`get_kinematic_geometry`), poses (`tf`), and optional
state-dependent extras (`get_dynamic_geometry`). Here is the simplest look for
the mass–spring–damper from §3 — ground, a box, and a spring coil:


In [ ]:
from minilink.core.kinematics import translation
from minilink.graphical.animation.primitives import Box, ground_line
from minilink.graphical.catalog.shapes import spring_between


def get_kinematic_geometry(self):
    return {
        "world": [ground_line(length=8.0)],
        "body": [
            Box(length_x=0.6, length_y=0.6, length_z=0.12, color="steelblue")
        ],
    }


def tf(self, x, u, t=0, params=None):
    return {"body": translation(x[0], 0.0, 0.0)}


def get_dynamic_geometry(self, x, u, t=0, params=None):
    # Spring stretches between a fixed wall and the left face of the mass.
    return {"world": [spring_between([-2.0, 0.0], [x[0] - 0.3, 0.0])]}


MassSpringDamper.get_kinematic_geometry = get_kinematic_geometry
MassSpringDamper.tf = tf
MassSpringDamper.get_dynamic_geometry = get_dynamic_geometry
msd.camera_scale = 4.0
msd_chain.camera_scale = 4.0


In [ ]:
msd_chain.animate(renderer="plotly")
# msd_chain.animate(renderer="meshcat")   # 3D in the browser
# msd_chain.animate(renderer="matplotlib")
# msd_chain.animate(renderer="pygame")    # native window (local)


## 8. Optimization

`minilink.optimization` solves finite-dimensional NLPs: define a
`MathematicalProgram` (`J`, optional constraints), then `Optimizer`.
Trajectory planning (§10) transcribes a `PlanningProblem` into the same layer.

Minimal unconstrained quadratic — min ½‖z − z̄‖², solution z* = z̄:


In [ ]:
from minilink.optimization.mathematical_program import MathematicalProgram
from minilink.optimization.optimizer import Optimizer

z_bar = np.array([1.0, -0.5, 2.0])

prog = MathematicalProgram(
    n_z=3,
    J=lambda z: 0.5 * np.dot(z - z_bar, z - z_bar),
    grad_J=lambda z: z - z_bar,
)

res = Optimizer(prog, z0=np.zeros(3), method=_OPTIMIZER_METHOD).solve(disp=True)


## 9. Compiled execution and JAX

Catalog JAX twins such as `JaxCartPole` compile to a flat execution plan.
`compile(backend="jax")` JIT-compiles **f** and keeps it traceable for
autodiff. Deeper story: [`showcase/jax.ipynb`](jax.ipynb).


In [ ]:
import time

import jax
from minilink.core.backends import configure_jax
from minilink.dynamics.catalog.pendulum.cartpole import JaxCartPole

configure_jax(enable_x64=True)

jax_cartpole = JaxCartPole()
jax_eval = jax_cartpole.compile(backend="jax")

n = 1000
xc = np.zeros(jax_cartpole.n)
uc = np.zeros(jax_cartpole.m)

t0 = time.perf_counter()
for _ in range(n):
    jax_cartpole.f(xc, uc, 0.0)
t_interp = time.perf_counter() - t0

t0 = time.perf_counter()
for _ in range(n):
    jax_eval.f(xc, uc, 0.0)
t_jit = time.perf_counter() - t0

print(f"JaxCartPole.f (interpreted) : {1e6 * t_interp / n:6.1f} µs/call")
print(f"compiled (jax jit)          : {1e6 * t_jit / n:6.1f} µs/call  ({t_interp / t_jit:.0f}× faster)")

A = jax.jacfwd(lambda x: jax_eval.f(x, uc, 0.0))(xc)
print("Linearization A = df/dx:\n", np.asarray(A).round(2))


## 10. Trajectory optimization

Cart-pole swing-up on `jax_cartpole` — `compile_backend="jax"` reuses the same
JIT-compiled **f** for dynamics defects and NLP gradients.


In [ ]:
from minilink.core.costs import QuadraticCost
from minilink.planning.problems import PlanningProblem
from minilink.planning.trajectory_optimization.planner import (
    TrajectoryOptimizationPlanner,
)

jax_cartpole.inputs["u"].lower_bound[0] = -10.0
jax_cartpole.inputs["u"].upper_bound[0] = 10.0

x_start = np.array([-2.0, 0.0, 0.0, 0.0])
x_goal = np.array([0.0, np.pi, 0.0, 0.0])

problem = PlanningProblem(
    sys=jax_cartpole,
    x_start=x_start,
    x_goal=x_goal,
    cost=QuadraticCost.from_system(
        jax_cartpole, Q=np.diag([1.0, 1.0, 0.0, 0.0]), xbar=x_goal
    ),
    tf=4.0,
)

planner = TrajectoryOptimizationPlanner(
    problem,
    n_steps=20,
    transcription="direct_collocation",
    compile_backend="jax",
    optimizer_method=_OPTIMIZER_METHOD,
    solve_disp=True,
)

traj = planner.solve().trajectory


In [ ]:
planner.plot_solution(signals=("x", "u"))


In [ ]:
jax_cartpole.animate(traj)


## 11. Hybrid and MPC

Continuous plants stay the core. Discrete control (digital MPC, sampled SMC, …)
uses a small parallel stack:

1. **`StepSystem`** — discrete leaf (tick logic); wire several into a **`StepDiagramSystem`** if needed
2. **`Computer`** — schedules that step side (`block % dt`)
3. **`HybridDiagram`** — `Computer @ plant` (or `mpc @ plant`) with ZOH + sampling; the continuous plant is solved between ticks

Below is the minimal closed-loop pattern: `ModelPredictiveController` then
`mpc @ plant`. More demos live under `examples/scripts/mpc/` (spatial circuit:
[mpc.ipynb](../applications/mpc.ipynb)).


In [ ]:
from minilink.control.mpc import (
    ModelPredictiveController,
    mpc_animation_overlays,
)
from minilink.core.backends import configure_jax
from minilink.core.costs import QuadraticCost
from minilink.dynamics.catalog.vehicles.jax_vehicles import BicycleDynRate
from minilink.planning.problems import PlanningProblem
from minilink.planning.trajectory_optimization.planner import (
    TrajectoryOptimizationPlanner,
)

configure_jax(enable_x64=True)

U_TARGET = 4.0
TF_SIM = 10.0
MPC_DT = 0.2
SIM_DT = 0.02

plant = BicycleDynRate()
r_r = plant.params["r_r"]
x_ref = np.array([0.0, 0.0, 0.0, U_TARGET, 0.0, 0.0, U_TARGET / r_r, 0.0])
x0 = np.array([0.0, 3.0, 0.0, U_TARGET * 0.8, 0.0, 0.0, (U_TARGET * 0.8) / r_r, 0.0])
plant.x0 = x0.copy()

mpc_planner = TrajectoryOptimizationPlanner(
    PlanningProblem(
        sys=plant,
        tf=2.0,
        x_start=x0,
        cost=QuadraticCost.from_system(
            plant,
            Q=np.diag([0.0, 12.0, 18.0, 0.5, 4.0, 6.0, 0.1, 100.0]),
            R=np.diag([1.0, 25.0]),
            S=np.diag([0.0, 30.0, 40.0, 2.0, 12.0, 18.0, 0.1, 100.0]),
            xbar=x_ref,
            ubar=np.zeros(2),
        ),
    ),
    n_steps=5,
    transcription="direct_collocation",
    compile_backend="jax",
    optimizer_method="scipy_slsqp",
    optimizer_options={"maxiter": 10, "ftol": 1.0},
)

mpc = ModelPredictiveController(
    mpc_planner, dt_mpc=MPC_DT, warm_start=True, step_disp=False
)



In [ ]:
hybrid = mpc @ plant
#  hybrid.plot_diagram()


In [ ]:
result = hybrid.compute_trajectory(
    tf=TF_SIM,
    x0_plant=x0,
    plant_dt_inner=SIM_DT,
    compile_backend="jax",
)


In [ ]:
hybrid.plot_trajectory()


In [ ]:
hybrid.animate(
    overlays=mpc_animation_overlays(result, mpc_planner, reference_pad=20.0)
)
